# GPT 사전학습 파이프라인 (Decoder-only)

## GPT 모델 구성 시 변경사항
# 데이터 블록

- 번역용 src/tgt 2시퀀스 입력에서, GPT용 단일 시퀀스 입력으로 변경
- 포맷을 <bos> 질문 <sep> 답변 <eos>로 통일
- 학습 타깃을 input_ids=seq[:-1], labels=seq[1:]의 next-token prediction으로 변경
- 패딩 라벨은 -100으로 마스킹하여 loss에서 제외


# 토크나이저 블록

- 번역쌍 코퍼스 기반에서 단일 대화 코퍼스 기반 학습으로 전환
- sep 특수 토큰 추가
- vocab 과대요청 오류 대응을 위해 hard_vocab_limit=false 및 재시도 로직 추가


# 모델 입력 블록

- Transformer 인코더 입력 경로 제거
- GPT 입력 형태로 변경:
- token embedding + learned positional embedding + dropout
- position_ids를 배치별 [0..T-1]로 생성해 위치 임베딩에 사용


# 어텐션/마스킹 블록

- 인코더 마스크/디코더-인코더 마스크 제거
- Decoder self-attention만 사용
- causal mask(미래 토큰 차단) + key padding mask 결합


# 트랜스포머 본체 블록

- Encoder-Decoder 구조에서 Decoder-only 스택으로 변경
- 블록 내부는 masked self-attention + FFN + residual/LayerNorm 유지
- 최종 출력은 LM head로 vocab logits 생성 (weight tying 적용)


# 학습 블록

- seq2seq teacher forcing 흐름에서 Causal LM loss 흐름으로 변경
- 학습 효율화를 위해 warmup+cosine LR scheduler 추가
- checkpoint 저장/로드(last.pt, best.pt) 및 이어학습(resume) 추가


# 검증/추론 블록

- 번역 BLEU 중심 검증에서 입력구성/shape/mask 검증 중심으로 전환
- 프롬프트 <bos> 질문 <sep> 기반 greedy 생성으로 동작 확인
- 체크포인트 상태 조회 셀을 추가해 재학습 시작지점(epoch/step) 확인 가능하게 구성

In [1]:
# ===== 라이브러리 불러오기 및 실행 환경 설정 =====
import csv
import io
import math
import os
import random
import re
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
from torch.utils.data import DataLoader, TensorDataset

# 실행 결과 재현을 위해 시드를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU 사용 가능하면 GPU를 사용하고, 아니면 CPU를 사용합니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch 버전:', torch.__version__)
print('사용 디바이스:', device)


PyTorch 버전: 2.7.1+cu118
사용 디바이스: cuda


## 1. 데이터 로딩 및 정규화
- 인코딩 fallback: `utf-8-sig -> cp949 -> euc-kr -> utf-8`
- 의미 손실을 줄이기 위해 공백 정리만 수행


In [2]:
def normalize_text(text: str) -> str:
    """문자열의 연속 공백을 하나로 정리하고 양 끝 공백을 제거합니다."""
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_chatbot_pairs(csv_path: str) -> List[Tuple[str, str]]:
    """Q/A 컬럼을 인코딩 fallback으로 읽고, 빈 행을 제거합니다."""
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    last_err = None

    for enc in encodings:
        try:
            rows: List[Tuple[str, str]] = []
            with open(csv_path, 'r', encoding=enc, newline='') as f:
                reader = csv.DictReader(f)
                fields = reader.fieldnames or []
                if 'Q' not in fields or 'A' not in fields:
                    raise ValueError(f"필수 컬럼(Q, A)이 없습니다. 현재 컬럼: {fields}")

                for row in reader:
                    q = normalize_text(row.get('Q', ''))
                    a = normalize_text(row.get('A', ''))
                    if not q or not a:
                        continue
                    rows.append((q, a))

            print(f'인코딩 {enc}로 {len(rows)}개 대화쌍을 불러왔습니다.')
            return rows
        except Exception as e:
            last_err = e

    raise RuntimeError(f'CSV 로딩에 실패했습니다. 마지막 오류: {last_err}')


pairs = load_chatbot_pairs('ChatbotData.csv')
print('샘플 대화쌍:', pairs[0])


인코딩 utf-8-sig로 11823개 대화쌍을 불러왔습니다.
샘플 대화쌍: ('12시 땡!', '하루가 또 가네요.')


## 2. 데이터 증강 (Lexical Substitution)
- `ko.zip`의 `ko.tsv` 임베딩을 파싱
- 질문/답변에서 치환 가능한 토큰 1개만 교체
- 치환이 불안정하면 해당 샘플은 건너뜀


In [3]:
def parse_vector(vec_text: str) -> Optional[np.ndarray]:
    """문자열 벡터를 float32 numpy 배열로 변환합니다."""
    cleaned = vec_text.replace('[', ' ').replace(']', ' ').strip()
    vec = np.fromstring(cleaned, sep=' ', dtype=np.float32)
    return vec if vec.size > 0 else None


def load_ko_embeddings_from_zip(
    zip_path: str,
    tsv_name: str = 'ko.tsv',
    max_words: int = 8000,
) -> Dict[str, np.ndarray]:
    """
    ko.tsv 벡터는 여러 줄에 걸쳐 있을 수 있어 누적 파싱합니다.
    첫 유효 벡터의 차원을 기준으로 차원이 다른 벡터는 제외합니다.
    """
    word2vec: Dict[str, np.ndarray] = {}
    target_dim: Optional[int] = None
    skipped_dim_mismatch = 0
    skipped_parse_fail = 0

    current_word: Optional[str] = None
    current_chunks: List[str] = []

    def flush_record():
        nonlocal current_word, current_chunks, target_dim
        nonlocal skipped_dim_mismatch, skipped_parse_fail

        if current_word is None or not current_chunks:
            current_word = None
            current_chunks = []
            return

        vec = parse_vector(' '.join(current_chunks))
        if vec is None:
            skipped_parse_fail += 1
        else:
            if target_dim is None:
                target_dim = int(vec.shape[0])
            if int(vec.shape[0]) != target_dim:
                skipped_dim_mismatch += 1
            elif current_word not in word2vec:
                word2vec[current_word] = vec

        current_word = None
        current_chunks = []

    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(tsv_name) as f:
            wrapper = io.TextIOWrapper(f, encoding='utf-8', errors='ignore')
            for raw_line in wrapper:
                line = raw_line.strip()
                if not line:
                    continue

                parts = line.split('\t', 2)

                # 새 레코드 시작: index\tword\t[vector...
                if len(parts) >= 3:
                    flush_record()
                    current_word = parts[1].strip()
                    current_chunks = [parts[2].strip()]
                    if ']' in parts[2]:
                        flush_record()

                # 현재 벡터의 continuation line
                elif current_word is not None:
                    current_chunks.append(line)
                    if ']' in line:
                        flush_record()

                if len(word2vec) >= max_words:
                    break

            flush_record()

    print('임베딩 로딩 요약')
    print(' - 로딩 단어 수:', len(word2vec))
    print(' - 벡터 차원:', target_dim)
    print(' - 차원 불일치로 제외:', skipped_dim_mismatch)
    print(' - 파싱 실패로 제외:', skipped_parse_fail)
    return word2vec


word2vec = load_ko_embeddings_from_zip('ko.zip', max_words=8000)
if len(word2vec) == 0:
    raise RuntimeError('ko.zip/ko.tsv에서 유효한 벡터를 찾지 못했습니다.')


임베딩 로딩 요약
 - 로딩 단어 수: 8000
 - 벡터 차원: 200
 - 차원 불일치로 제외: 0
 - 파싱 실패로 제외: 0


In [4]:
def build_embedding_matrix(word2vec: Dict[str, np.ndarray]) -> Tuple[List[str], np.ndarray]:
    """코사인 유사도 계산을 위해 임베딩 행렬을 정규화합니다."""
    words = list(word2vec.keys())
    mat = np.stack([word2vec[w] for w in words], axis=0).astype(np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-8
    return words, mat / norms


emb_words, emb_mat = build_embedding_matrix(word2vec)
emb_index = {w: i for i, w in enumerate(emb_words)}


def find_similar_word(
    token: str,
    top_k: int = 8,
    sim_threshold: float = 0.45,
    candidate_pool: int = 2048,
) -> Optional[str]:
    """랜덤 후보군 기반 근사 탐색으로 유사 단어를 찾습니다."""
    if token not in emb_index:
        return None

    idx = emb_index[token]
    vec = emb_mat[idx]

    vocab_n = emb_mat.shape[0]
    if vocab_n <= candidate_pool:
        candidate_idx = np.arange(vocab_n)
    else:
        candidate_idx = np.random.choice(vocab_n, size=candidate_pool, replace=False)

    candidate_idx = candidate_idx[candidate_idx != idx]
    if candidate_idx.size == 0:
        return None

    sims = emb_mat[candidate_idx] @ vec
    rank = np.argsort(-sims)

    for r in rank[:top_k]:
        if sims[r] < sim_threshold:
            continue
        cand = emb_words[int(candidate_idx[r])]
        if cand != token and len(cand) >= 2:
            return cand
    return None


def lexical_substitute_text(text: str, replace_prob: float = 0.4) -> Optional[str]:
    """문장 내 치환 가능한 토큰 1개를 유사 단어로 바꿉니다."""
    tokens = text.split()
    if len(tokens) == 0 or random.random() > replace_prob:
        return None

    valid_positions = [i for i, t in enumerate(tokens) if t in emb_index and len(t) >= 2]
    if not valid_positions:
        return None

    pos = random.choice(valid_positions)
    replacement = find_similar_word(tokens[pos])
    if replacement is None:
        return None

    tokens[pos] = replacement
    out = ' '.join(tokens).strip()
    return out if out != text else None


def augment_pairs(original_pairs: List[Tuple[str, str]], augment_ratio: float = 0.20) -> List[Tuple[str, str]]:
    """원본 + 증강 샘플을 혼합하고 중복을 제거합니다."""
    mixed = list(original_pairs)
    target_n = int(len(original_pairs) * augment_ratio)
    sampled = random.sample(original_pairs, k=min(target_n, len(original_pairs)))

    for q, a in sampled:
        new_q = lexical_substitute_text(q)
        new_a = lexical_substitute_text(a)
        if new_q is None and new_a is None:
            continue
        mixed.append((new_q if new_q is not None else q, new_a if new_a is not None else a))

    return list(dict.fromkeys(mixed))


mixed_pairs = augment_pairs(pairs, augment_ratio=0.20)
print('원본 대화쌍 수:', len(pairs))
print('증강 혼합 후 대화쌍 수:', len(mixed_pairs))


원본 대화쌍 수: 11823
증강 혼합 후 대화쌍 수: 12303


## 3. 토크나이저 학습 및 Causal LM 샘플 생성
- 시퀀스 형식: `<bos> 질문 <sep> 답변 <eos>`
- 길이 정책: `block_size + 1` 기준으로 truncate/pad
- shift: `input_ids=seq[:-1]`, `labels=seq[1:]`


In [5]:
SPECIAL_SEP = '<sep>'
VOCAB_SIZE = 8000
BLOCK_SIZE = 64


def train_sentencepiece(corpus_texts: List[str], model_prefix: str, vocab_size: int) -> spm.SentencePieceProcessor:
    """
    SentencePiece 학습 함수
    - hard_vocab_limit=false로 과도한 vocab 요청 실패를 줄입니다.
    - 실패 시 vocab_size를 줄여 재시도합니다.
    """
    corpus_file = f'{model_prefix}_corpus.txt'
    with open(corpus_file, 'w', encoding='utf-8') as f:
        for line in corpus_texts:
            f.write(line + '\n')

    current_vocab = int(vocab_size)
    min_vocab = 1000

    while current_vocab >= min_vocab:
        try:
            args = [
                f'--input={corpus_file}',
                f'--model_prefix={model_prefix}',
                f'--vocab_size={current_vocab}',
                '--model_type=unigram',
                '--character_coverage=0.9995',
                '--pad_id=0',
                '--bos_id=1',
                '--eos_id=2',
                '--unk_id=3',
                f'--user_defined_symbols={SPECIAL_SEP}',
                '--input_sentence_size=1000000',
                '--shuffle_input_sentence=true',
                '--hard_vocab_limit=false',
            ]
            spm.SentencePieceTrainer.Train(' '.join(args))

            sp = spm.SentencePieceProcessor()
            sp.Load(f'{model_prefix}.model')
            print(f'SentencePiece 학습 성공 (요청 vocab_size={current_vocab})')
            print(f'실제 vocab 크기: {sp.GetPieceSize()}')
            return sp

        except RuntimeError as e:
            print(f'[재시도] vocab_size={current_vocab} 실패: {e}')
            current_vocab = int(current_vocab * 0.85)

    raise RuntimeError('SentencePiece 학습이 모든 재시도에서 실패했습니다.')


train_texts = [f"{q} {SPECIAL_SEP} {a}" for q, a in mixed_pairs]
tokenizer = train_sentencepiece(train_texts, model_prefix='gpt_ko_chatbot_spm', vocab_size=VOCAB_SIZE)

PAD_ID = tokenizer.pad_id()
BOS_ID = tokenizer.bos_id()
EOS_ID = tokenizer.eos_id()
SEP_ID = tokenizer.piece_to_id(SPECIAL_SEP)
print('특수 토큰 ID:', {'pad': PAD_ID, 'bos': BOS_ID, 'eos': EOS_ID, 'sep': SEP_ID})


SentencePiece 학습 성공 (요청 vocab_size=8000)
실제 vocab 크기: 8000
특수 토큰 ID: {'pad': 0, 'bos': 1, 'eos': 2, 'sep': 4}


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=gpt_ko_chatbot_spm_corpus.txt --model_prefix=gpt_ko_chatbot_spm --vocab_size=8000 --model_type=unigram --character_coverage=0.9995 --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 --user_defined_symbols=<sep> --input_sentence_size=1000000 --shuffle_input_sentence=true --hard_vocab_limit=false
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: gpt_ko_chatbot_spm_corpus.txt
  input_format: 
  model_prefix: gpt_ko_chatbot_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 1000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  all

In [6]:
def build_sequence_ids(q: str, a: str, sp: spm.SentencePieceProcessor, block_size: int) -> List[int]:
    """<bos> Q <sep> A <eos> 형태의 고정 길이 시퀀스를 생성합니다."""
    q_ids = sp.EncodeAsIds(q)
    a_ids = sp.EncodeAsIds(a)
    seq = [BOS_ID] + q_ids + [SEP_ID] + a_ids + [EOS_ID]

    max_total = block_size + 1
    if len(seq) > max_total:
        seq = seq[:max_total]
        seq[-1] = EOS_ID
    else:
        seq = seq + [PAD_ID] * (max_total - len(seq))
    return seq


def build_causal_lm_tensors(chat_pairs: List[Tuple[str, str]], sp: spm.SentencePieceProcessor, block_size: int):
    """GPT Causal LM 학습용 input_ids, labels, attention_mask를 만듭니다."""
    all_input_ids, all_labels, all_attention = [], [], []

    for q, a in chat_pairs:
        seq = build_sequence_ids(q, a, sp, block_size)
        input_ids = seq[:-1]
        labels = seq[1:]

        attention_mask = [0 if tok == PAD_ID else 1 for tok in input_ids]
        labels = [tok if tok != PAD_ID else -100 for tok in labels]

        all_input_ids.append(input_ids)
        all_labels.append(labels)
        all_attention.append(attention_mask)

    return (
        torch.tensor(all_input_ids, dtype=torch.long),
        torch.tensor(all_labels, dtype=torch.long),
        torch.tensor(all_attention, dtype=torch.long),
    )


input_ids, labels, attention_mask = build_causal_lm_tensors(mixed_pairs, tokenizer, BLOCK_SIZE)
print('input_ids shape:', tuple(input_ids.shape))
print('labels shape:', tuple(labels.shape))
print('attention_mask shape:', tuple(attention_mask.shape))


input_ids shape: (12303, 64)
labels shape: (12303, 64)
attention_mask shape: (12303, 64)


## 4. 모델 구성: GPT 입력 블록과 Decoder 스택
- 입력: token embedding + learned positional embedding + dropout
- 마스킹: causal mask + key padding mask
- 출력: LM head (weight tying 적용)


In [7]:
class GPTBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln_2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor, key_padding_mask: Optional[torch.Tensor]) -> torch.Tensor:
        # 1) Masked self-attention + residual
        h = self.ln_1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + self.dropout(attn_out)

        # 2) Feed-forward + residual
        h = self.ln_2(x)
        x = x + self.ff(h)
        return x


@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 8
    d_ff: int = 1024
    dropout: float = 0.1


class GPTLanguageModel(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        # GPT 입력 블록: 토큰 임베딩 + 위치 임베딩
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

        # Decoder block 스택
        self.blocks = nn.ModuleList([
            GPTBlock(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout) for _ in range(cfg.n_layers)
        ])
        self.ln_f = nn.LayerNorm(cfg.d_model)

        # LM head + weight tying
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def _build_causal_mask(self, t: int, device: torch.device) -> torch.Tensor:
        """미래 토큰을 가리는 상삼각 causal mask를 생성합니다."""
        return torch.triu(torch.ones(t, t, device=device, dtype=torch.bool), diagonal=1)

    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, labels: Optional[torch.Tensor] = None):
        bsz, seqlen = input_ids.shape
        assert seqlen <= self.cfg.block_size, 'sequence length exceeds block_size'

        # position_ids: [0..T-1]을 배치 크기만큼 확장
        position_ids = torch.arange(seqlen, dtype=torch.long, device=input_ids.device).unsqueeze(0).expand(bsz, -1)

        # 입력 임베딩 결합
        x = self.tok_emb(input_ids) + self.pos_emb(position_ids)
        x = self.drop(x)

        # attention mask 구성
        attn_mask = self._build_causal_mask(seqlen, input_ids.device)
        key_padding_mask = (attention_mask == 0) if attention_mask is not None else None

        # Decoder 블록 통과
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=key_padding_mask)

        logits = self.lm_head(self.ln_f(x))

        # labels가 있으면 Causal LM 손실 계산
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)

        return logits, loss, position_ids, attn_mask


In [8]:
# ===== 데이터로더, 모델, 옵티마이저 준비 =====
BATCH_SIZE = 64
BASE_LR = 3e-4

dataset = TensorDataset(input_ids, attention_mask, labels)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

cfg = GPTConfig(
    vocab_size=tokenizer.GetPieceSize(),
    block_size=BLOCK_SIZE,
    d_model=256,
    n_layers=4,
    n_heads=8,
    d_ff=1024,
    dropout=0.1,
)

model = GPTLanguageModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR)

print("vocab_size:", cfg.vocab_size)
print("배치 개수:", len(dataloader))
print("초기 학습률(BASE_LR):", BASE_LR)


vocab_size: 8000
배치 개수: 193
초기 학습률(BASE_LR): 0.0003


## 5. 입력 검증: Position IDs, Mask, Shape
- 샘플 입력/정답/마스크 확인
- 모델 출력 shape 확인
- causal mask 상삼각 구조 확인


In [9]:
# ===== 샘플 입력 구조 확인 =====
sample_idx = 0
sample_input = input_ids[sample_idx]
sample_label = labels[sample_idx]
sample_attn = attention_mask[sample_idx]

print('샘플 input_ids[:25]:', sample_input[:25].tolist())
print('샘플 labels[:25]:   ', sample_label[:25].tolist())
print('샘플 attn[:25]:     ', sample_attn[:25].tolist())

# 사람이 읽기 쉽게 디코딩하여 확인
reconstructed_seq = sample_input.tolist() + [sample_label[-1].item() if sample_label[-1].item() != -100 else PAD_ID]
readable_ids = [x for x in reconstructed_seq if x not in [PAD_ID, -100]]
print('디코딩 샘플(일부):', tokenizer.DecodeIds(readable_ids)[:200])

# 미니배치 forward 검증
batch_input, batch_attn, batch_labels = next(iter(dataloader))
batch_input = batch_input.to(device)
batch_attn = batch_attn.to(device)
batch_labels = batch_labels.to(device)

logits, loss, position_ids, causal_mask = model(batch_input, attention_mask=batch_attn, labels=batch_labels)
print('logits shape:', tuple(logits.shape))
print('loss:', float(loss.detach().cpu()))
print('position_ids shape:', tuple(position_ids.shape))
print('causal_mask shape:', tuple(causal_mask.shape))

# 과제 요구 shape 검증
assert batch_input.ndim == 2
assert position_ids.shape == batch_input.shape
assert logits.ndim == 3
assert logits.shape[:2] == batch_input.shape
assert logits.shape[-1] == cfg.vocab_size
print('shape 검증 통과')


샘플 input_ids[:25]: [1, 4338, 329, 5, 7832, 68, 4, 253, 7, 100, 5, 7, 23, 6, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
샘플 labels[:25]:    [4338, 329, 5, 7832, 68, 4, 253, 7, 100, 5, 7, 23, 6, 2, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
샘플 attn[:25]:      [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
디코딩 샘플(일부): 12시 땡!<sep> 하루가 또 가네요.
logits shape: (64, 64, 8000)
loss: 156.37107849121094
position_ids shape: (64, 64)
causal_mask shape: (64, 64)
shape 검증 통과


In [10]:
# ===== causal mask 시각 확인 (1이면 마스킹) =====
T = 12
small_mask = torch.triu(torch.ones(T, T, dtype=torch.int64), diagonal=1)
print('기대되는 상삼각 causal mask (1=가림):')
print(small_mask)


기대되는 상삼각 causal mask (1=가림):
tensor([[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


## 6. 학습 동작 확인 (Sanity Run)
- 파이프라인 동작 확인용 짧은 학습


## 6-1. 학습 효율화: Learning Rate Scheduler + Checkpoint
- 스케줄러: Warmup + Cosine Decay
- 체크포인트: 주기 저장 + 마지막 상태 저장 + 학습 재개


In [23]:
# ===== 스케줄러/체크포인트 설정 =====
EPOCHS = 10
MAX_STEPS = 5000
WARMUP_STEPS = 20
MIN_LR_RATIO = 0.10
SAVE_EVERY_STEPS = 500
RESUME_TRAINING = True

CKPT_DIR = Path("./checkpoints_gpt")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LAST_CKPT_PATH = CKPT_DIR / "last.pt"
BEST_CKPT_PATH = CKPT_DIR / "best.pt"


def build_warmup_cosine_scheduler(
    optimizer: torch.optim.Optimizer,
    warmup_steps: int,
    total_steps: int,
    min_lr_ratio: float = 0.1,
):
    """초기 warmup 이후 cosine decay를 적용하는 LambdaLR 스케줄러."""
    warmup_steps = max(1, int(warmup_steps))
    total_steps = max(warmup_steps + 1, int(total_steps))

    def lr_lambda(current_step: int) -> float:
        # 1) warmup 구간: 선형 증가
        if current_step < warmup_steps:
            return float(current_step + 1) / float(warmup_steps)

        # 2) decay 구간: cosine 감소
        progress = (current_step - warmup_steps) / float(total_steps - warmup_steps)
        progress = min(max(progress, 0.0), 1.0)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    epoch: int,
    global_step: int,
    loss_hist: List[float],
    best_loss: float,
):
    """학습 상태를 저장합니다."""
    state = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "epoch": epoch,
        "global_step": global_step,
        "loss_hist": loss_hist,
        "best_loss": best_loss,
        "config": cfg.__dict__,
    }
    torch.save(state, path)
    print(f"[체크포인트 저장] {path} | step={global_step} | best_loss={best_loss:.4f}")


def load_checkpoint_if_exists(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
):
    """체크포인트가 있으면 불러오고, 없으면 초기 상태로 시작합니다."""
    if not path.exists():
        print("[체크포인트] 기존 파일 없음. 처음부터 학습을 시작합니다.")
        return 0, 0, [], float("inf")

    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    start_epoch = int(ckpt.get("epoch", 0))
    global_step = int(ckpt.get("global_step", 0))
    loss_hist = list(ckpt.get("loss_hist", []))
    best_loss = float(ckpt.get("best_loss", float("inf")))

    print(f"[체크포인트 로드] {path} | epoch={start_epoch} | step={global_step} | best_loss={best_loss:.4f}")
    return start_epoch, global_step, loss_hist, best_loss


total_train_steps = max(MAX_STEPS, EPOCHS * len(dataloader))
scheduler = build_warmup_cosine_scheduler(
    optimizer=optimizer,
    warmup_steps=WARMUP_STEPS,
    total_steps=total_train_steps,
    min_lr_ratio=MIN_LR_RATIO,
)

print("스케줄러 설정 완료")
print(" - total_train_steps:", total_train_steps)
print(" - warmup_steps:", WARMUP_STEPS)
print(" - min_lr_ratio:", MIN_LR_RATIO)
print(" - checkpoint dir:", CKPT_DIR)


스케줄러 설정 완료
 - total_train_steps: 5000
 - warmup_steps: 20
 - min_lr_ratio: 0.1
 - checkpoint dir: checkpoints_gpt


## 6-2. 체크포인트 상태 확인 (재학습 시작점 판단)
- 이어학습 전에 마지막 저장 지점(epoch/step/loss)을 먼저 확인합니다.

In [28]:
# ===== 체크포인트 상태 조회 =====
def inspect_checkpoint(path: Path):
    """체크포인트 파일에서 이어학습 판단에 필요한 정보를 출력합니다."""
    if not path.exists():
        print(f'[체크포인트 없음] {path}')
        return None

    ckpt = torch.load(path, map_location='cpu')
    epoch = int(ckpt.get('epoch', -1))
    step = int(ckpt.get('global_step', 0))
    best_loss = float(ckpt.get('best_loss', float('inf')))
    hist = ckpt.get('loss_hist', [])
    last_loss = float(hist[-1]) if len(hist) > 0 else None

    print(f'[체크포인트 정보] {path.name}')
    print(' - 마지막 epoch:', epoch)
    print(' - 누적 step:', step)
    print(' - 마지막 loss:', last_loss)
    print(' - best loss:', best_loss)

    # 실무적으로는 다음 epoch부터 이어서 학습
    recommended_next_epoch = epoch + 1 if epoch >= 0 else 0
    print(' - 이어학습 권장 시작 epoch:', recommended_next_epoch)
    return {
        'epoch': epoch,
        'global_step': step,
        'last_loss': last_loss,
        'best_loss': best_loss,
        'recommended_next_epoch': recommended_next_epoch,
    }

last_info = inspect_checkpoint(LAST_CKPT_PATH)
best_info = inspect_checkpoint(BEST_CKPT_PATH)


[체크포인트 정보] last.pt
 - 마지막 epoch: 9
 - 누적 step: 2158
 - 마지막 loss: 7.298163890838623
 - best loss: 6.560866355895996
 - 이어학습 권장 시작 epoch: 10
[체크포인트 정보] best.pt
 - 마지막 epoch: 9
 - 누적 step: 2056
 - 마지막 loss: 6.560866355895996
 - best loss: 6.560866355895996
 - 이어학습 권장 시작 epoch: 10


In [25]:
# ===== 짧은 학습 루프 (스케줄러 + 체크포인트) =====
start_epoch = 0
global_step = 0
loss_hist = []
best_loss = float("inf")

if RESUME_TRAINING:
    start_epoch, global_step, loss_hist, best_loss = load_checkpoint_if_exists(
        LAST_CKPT_PATH,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
    )

model.train()
stop_training = False
last_epoch = start_epoch - 1

for epoch in range(start_epoch, EPOCHS):
    last_epoch = epoch
    for b_input, b_attn, b_labels in dataloader:
        b_input = b_input.to(device)
        b_attn = b_attn.to(device)
        b_labels = b_labels.to(device)

        optimizer.zero_grad()
        _, loss, _, _ = model(b_input, attention_mask=b_attn, labels=b_labels)
        loss.backward()
        optimizer.step()
        scheduler.step()  # 스텝 단위 LR 스케줄 적용

        current_loss = float(loss.detach().cpu())
        loss_hist.append(current_loss)
        global_step += 1

        # best 모델 갱신
        if current_loss < best_loss:
            best_loss = current_loss
            save_checkpoint(
                BEST_CKPT_PATH,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=last_epoch,
                global_step=global_step,
                loss_hist=loss_hist,
                best_loss=best_loss,
            )

        # 주기적 체크포인트 저장
        if global_step % SAVE_EVERY_STEPS == 0:
            save_checkpoint(
                LAST_CKPT_PATH,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=last_epoch,
                global_step=global_step,
                loss_hist=loss_hist,
                best_loss=best_loss,
            )

        if global_step % 20 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"학습 step {global_step:4d} | loss {current_loss:.4f} | lr {current_lr:.8f}")

        if global_step >= MAX_STEPS:
            stop_training = True
            break

    if stop_training:
        break

# 마지막 상태 저장
save_checkpoint(
    LAST_CKPT_PATH,
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    epoch=last_epoch,
    global_step=global_step,
    loss_hist=loss_hist,
    best_loss=best_loss,
)

print("학습 완료. 마지막 loss:", loss_hist[-1] if loss_hist else None)
print("최저 loss(best_loss):", best_loss)
print("최종 step:", global_step)


[체크포인트 로드] checkpoints_gpt/last.pt | epoch=4 | step=1000 | best_loss=7.7589
[체크포인트 저장] checkpoints_gpt/best.pt | step=1013 | best_loss=7.7002
학습 step 1020 | loss 8.2188 | lr 0.00027300
학습 step 1040 | loss 8.0707 | lr 0.00027197
학습 step 1060 | loss 8.0504 | lr 0.00027093
학습 step 1080 | loss 7.9504 | lr 0.00026986
[체크포인트 저장] checkpoints_gpt/best.pt | step=1096 | best_loss=7.6036
학습 step 1100 | loss 7.7403 | lr 0.00026878
학습 step 1120 | loss 7.6969 | lr 0.00026768
학습 step 1140 | loss 8.0155 | lr 0.00026657
학습 step 1160 | loss 7.7948 | lr 0.00026544
[체크포인트 저장] checkpoints_gpt/best.pt | step=1169 | best_loss=7.5698
[체크포인트 저장] checkpoints_gpt/best.pt | step=1170 | best_loss=7.3452
학습 step 1180 | loss 7.7055 | lr 0.00026429
학습 step 1200 | loss 7.8725 | lr 0.00026313
학습 step 1220 | loss 7.4855 | lr 0.00026195
학습 step 1240 | loss 7.7027 | lr 0.00026076
학습 step 1260 | loss 7.6692 | lr 0.00025955
학습 step 1280 | loss 7.6446 | lr 0.00025833
[체크포인트 저장] checkpoints_gpt/best.pt | step=1288 | best_loss

## 7. 생성 동작 확인 (Greedy Decoding)
- 프롬프트: `<bos> 질문 <sep>`
- 종료: `eos` 또는 최대 생성 길이


In [29]:
@torch.no_grad()
def generate_answer(model: GPTLanguageModel, sp: spm.SentencePieceProcessor, question: str, max_new_tokens: int = 40) -> str:
    """질문 프롬프트를 입력해 답변 토큰을 Greedy 방식으로 생성합니다."""
    model.eval()

    # 프롬프트는 질문까지만 구성
    prompt_ids = [BOS_ID] + sp.EncodeAsIds(normalize_text(question)) + [SEP_ID]
    prompt_ids = prompt_ids[: model.cfg.block_size]
    generated = prompt_ids[:]

    # 토큰을 한 개씩 순차 생성
    for _ in range(max_new_tokens):
        x = torch.tensor([generated], dtype=torch.long, device=device)
        attn = torch.ones_like(x, dtype=torch.long, device=device)
        logits, _, _, _ = model(x, attention_mask=attn, labels=None)
        temperature = 0.8
        top_k = 50
        
        # next_logits = logits[0, -1] / temperature
        # topk_vals, topk_idx = torch.topk(next_logits, k=min(top_k, next_logits.size(-1)))
        # probs = torch.softmax(topk_vals, dim=-1)
        # sampled = torch.multinomial(probs, num_samples=1)
        # next_id = int(topk_idx[sampled].item())
        
        next_id = int(torch.argmax(logits[0, -1], dim=-1).item())
        generated.append(next_id)

        if len(generated) >= model.cfg.block_size or next_id == EOS_ID:
            break

    # <sep> 이후 구간만 답변으로 사용
    if SEP_ID in generated:
        sep_pos = generated.index(SEP_ID)
        answer_ids = []
        for tid in generated[sep_pos + 1:]:
            if tid in (EOS_ID, PAD_ID):
                break
            answer_ids.append(tid)
        print(answer_ids)
    else:
        answer_ids = generated

    return sp.DecodeIds(answer_ids).strip()


# 샘플 질문 2개로 생성 테스트
test_questions = [pairs[0][0], pairs[min(100, len(pairs) - 1)][0]]
for i, q in enumerate(test_questions, 1):
    ans = generate_answer(model, tokenizer, q, max_new_tokens=40)
    print(f'[{i}] 질문: {q}')
    print(f'    생성 답변: {ans}')


[5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 7, 7, 5, 13, 4, 4, 5, 13, 6]
[1] 질문: 12시 땡!
    생성 답변: 게 게 게 게 게 게 게 게 게 게 게 게 게 가가 게<sep><sep> 게.
[5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 7, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 5, 13, 4, 5, 13, 5, 13, 5]
[2] 질문: 거지됐어
    생성 답변: 게 게 게 게 게 게 게 가 게 게 게 게 게 게 게 게 게<sep> 게 게
